# 실습 5: 한글 TF-IDF 음식 메뉴 추천 시스템

맛집 메뉴 CSV 데이터를 이용해 사용자의 자연어 입력에 맞는 메뉴를 추천하는 실습입니다.

예시 입력:

- 해장 잘되는 음식
- 칼칼한 음식
- 다이어트에 좋은 음식
- 고소하고 부드러운 음식

이번 노트북은 수업용 Python/Jupyter 버전입니다. 같은 로직을 Next.js로 옮긴 서비스 화면은 `nextjs-menu-recommender/` 폴더에 있습니다.

## 1. 실습 목표

1. CSV 메뉴 데이터를 읽는다.
2. 메뉴명, 카테고리, 설명을 하나의 문서처럼 다룬다.
3. 사용자 입력 문장을 간단히 토큰화한다.
4. TF-IDF와 코사인 유사도로 입력과 메뉴 설명의 유사도를 계산한다.
5. `해장`, `칼칼함`, `다이어트` 같은 의도 키워드를 확장해 추천 품질을 보정한다.
6. 추천 결과를 표 형태로 확인한다.

In [ ]:
from pathlib import Path
import csv
import math
import re
from collections import Counter

DATA_PATH = Path("../nextjs-menu-recommender/data/menu-data.csv")
DATA_PATH.resolve()

## 2. CSV 데이터 읽기

pandas가 없어도 실행될 수 있도록 Python 표준 라이브러리인 `csv`로 읽습니다.

In [ ]:
def load_menu_data(path):
    with open(path, encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
    return rows

menus = load_menu_data(DATA_PATH)
print("메뉴 개수:", len(menus))
print("컬럼:", list(menus[0].keys()))
menus[:5]

## 3. 데이터 빠르게 확인하기

카테고리별 메뉴 수와 전체 메뉴명을 확인합니다.

In [ ]:
category_counts = Counter(row["카테고리"] for row in menus)
print("카테고리별 메뉴 수")
for category, count in category_counts.items():
    print(f"- {category}: {count}개")

print()
print("메뉴 목록")
for row in menus:
    print(f"- {row['메뉴명']} ({row['카테고리']})")

## 4. 간단한 한국어 토큰화

형태소 분석기 없이 실습하기 위해 간단한 규칙을 사용합니다.

- 한글, 영문, 숫자만 추출한다.
- 너무 짧은 토큰은 제거한다.
- 자주 등장하지만 의미가 약한 불용어를 제거한다.
- `다이어트에`, `칼칼한`처럼 붙은 조사/어미를 간단히 제거한다.

In [ ]:
STOPWORDS = {
    "음식", "메뉴", "추천", "좋은", "먹고", "싶어", "먹을", "만한",
    "오늘", "약간", "좀", "있는", "없는", "으로", "하고", "되는",
}

def normalize_korean_token(token):
    token = re.sub(r"(으로|에서|에게|에는|으로는|하고)$", "", token)
    token = re.sub(r"(에|은|는|이|가|을|를)$", "", token)
    token = re.sub(r"(스러운|스럽|하고|하게|한)$", "", token)
    return token

def tokenize_korean(text):
    text = str(text).lower()
    tokens = re.findall(r"[가-힣]+|[a-zA-Z]+|\d+(?:\.\d+)?", text)
    normalized = []
    for token in tokens:
        if re.search(r"[가-힣]", token):
            token = normalize_korean_token(token)
        if len(token) <= 1:
            continue
        if token in STOPWORDS:
            continue
        normalized.append(token)
    return normalized

examples = ["해장 잘되는 음식", "칼칼한 음식", "다이어트에 좋은 음식"]
for example in examples:
    print(example, "->", tokenize_korean(example))

## 5. 의도 키워드 확장

사용자가 `해장 잘되는 음식`이라고 입력했을 때 실제 메뉴 설명에는 `해장`이라는 단어가 없을 수 있습니다.

그래서 `해장 -> 국물, 얼큰, 칼칼, 시원, 육수, 찌개, 탕`처럼 도메인 키워드를 확장합니다.

In [ ]:
INTENT_RULES = [
    {
        "intent": "해장",
        "triggers": ["해장", "숙취", "술", "국물", "시원", "뜨끈", "따뜻"],
        "expansions": ["국물", "얼큰", "칼칼", "시원", "육수", "찌개", "탕", "라면", "국수"],
    },
    {
        "intent": "칼칼함",
        "triggers": ["칼칼", "얼큰", "매운", "맵", "매콤", "사천", "자극"],
        "expansions": ["칼칼", "얼큰", "매콤", "고추장", "두반장", "짬뽕", "김치", "떡볶이"],
    },
    {
        "intent": "다이어트",
        "triggers": ["다이어트", "가벼운", "가볍", "건강", "채소", "담백", "깔끔"],
        "expansions": ["신선", "나물", "채소", "가볍", "담백", "쌀국수", "비빔밥", "상큼"],
    },
    {
        "intent": "든든함",
        "triggers": ["든든", "배부른", "고기", "보양", "힘", "단백질"],
        "expansions": ["고기", "소갈비", "돼지고기", "진한", "탕", "삼겹살", "돈카츠"],
    },
    {
        "intent": "고소함",
        "triggers": ["고소", "크림", "치즈", "부드러운", "느끼", "꾸덕"],
        "expansions": ["고소", "치즈", "크림", "버터", "모짜렐라", "까르보나라", "리조또"],
    },
    {
        "intent": "상큼함",
        "triggers": ["상큼", "새콤", "개운", "깔끔", "산뜻"],
        "expansions": ["상큼", "새콤", "고수", "숙주", "쌀국수", "팟타이", "토마토"],
    },
]

def expand_query_tokens(query, tokens):
    expanded = set(tokens)
    matched_intents = []
    for rule in INTENT_RULES:
        matched = any(trigger in query for trigger in rule["triggers"])
        if not matched:
            continue
        matched_intents.append(rule["intent"])
        for token in rule["expansions"]:
            expanded.add(token)
    return list(expanded), matched_intents

query = "해장 잘되는 음식"
tokens = tokenize_korean(query)
expanded_tokens, matched_intents = expand_query_tokens(query, tokens)
print("원본 토큰:", tokens)
print("확장 토큰:", expanded_tokens)
print("감지된 의도:", matched_intents)

## 6. TF-IDF 직접 구현하기

수업 이해를 위해 scikit-learn 없이 TF-IDF를 직접 계산합니다.

In [ ]:
def term_frequency(tokens):
    return Counter(tokens)

def build_tfidf_vectors(token_docs):
    doc_count = len(token_docs)
    vocabulary = sorted(set(token for doc in token_docs for token in doc))
    document_frequency = {}
    for term in vocabulary:
        document_frequency[term] = sum(1 for doc in token_docs if term in doc)

    vectors = []
    for tokens in token_docs:
        tf = term_frequency(tokens)
        vector = {}
        for term in vocabulary:
            count = tf.get(term, 0)
            if count == 0:
                continue
            term_tf = count / max(len(tokens), 1)
            df = document_frequency.get(term, 0)
            idf = math.log((doc_count + 1) / (df + 1)) + 1
            vector[term] = term_tf * idf
        vectors.append(vector)
    return vectors

def cosine_similarity(vec_a, vec_b):
    dot = 0
    norm_a = 0
    norm_b = 0
    for value in vec_a.values():
        norm_a += value * value
    for value in vec_b.values():
        norm_b += value * value
    for term, value in vec_a.items():
        dot += value * vec_b.get(term, 0)
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot / ((norm_a ** 0.5) * (norm_b ** 0.5))

## 7. 의도 보정 점수

TF-IDF만 사용하면 `해장`처럼 설명에 직접 등장하지 않는 의도가 약해질 수 있습니다. 입력 의도와 메뉴 설명이 잘 맞으면 소량의 보정 점수를 더합니다.

In [ ]:
def get_intent_score(query, menu):
    text = f"{menu['메뉴명']} {menu['카테고리']} {menu['설명']}"
    matched_intents = []
    score = 0
    for rule in INTENT_RULES:
        if not any(trigger in query for trigger in rule["triggers"]):
            continue
        hits = [keyword for keyword in rule["expansions"] if keyword in text]
        if not hits:
            continue
        matched_intents.append(rule["intent"])
        score += min(0.35, len(hits) * 0.08)
    return min(score, 0.6), matched_intents

## 8. 메뉴 추천 함수 만들기

In [ ]:
def recommend_menus(query, menus, top_n=8):
    safe_query = query.strip() or "해장 잘되는 음식"
    query_tokens = tokenize_korean(safe_query)
    expanded_tokens, query_intents = expand_query_tokens(safe_query, query_tokens)

    menu_texts = [f"{row['메뉴명']} {row['카테고리']} {row['설명']}" for row in menus]
    token_docs = [expanded_tokens] + [tokenize_korean(text) for text in menu_texts]
    vectors = build_tfidf_vectors(token_docs)
    query_vector = vectors[0]
    menu_vectors = vectors[1:]

    results = []
    for menu, menu_vector, menu_tokens in zip(menus, menu_vectors, token_docs[1:]):
        tfidf_score = cosine_similarity(query_vector, menu_vector)
        intent_score, matched_intents = get_intent_score(safe_query, menu)
        final_score = min(1, tfidf_score * 0.78 + intent_score)
        results.append({
            "메뉴명": menu["메뉴명"],
            "카테고리": menu["카테고리"],
            "설명": menu["설명"],
            "최종점수": final_score,
            "TFIDF점수": tfidf_score,
            "의도점수": intent_score,
            "매칭의도": ", ".join(matched_intents) if matched_intents else "TF-IDF",
            "토큰": menu_tokens,
        })

    results.sort(key=lambda row: row["최종점수"], reverse=True)
    return {
        "query": safe_query,
        "query_tokens": query_tokens,
        "expanded_tokens": expanded_tokens,
        "query_intents": query_intents,
        "results": results[:top_n],
    }

## 9. 추천 결과 확인하기

In [ ]:
def print_recommendations(query):
    output = recommend_menus(query, menus, top_n=5)
    print("입력:", output["query"])
    print("입력 토큰:", output["query_tokens"])
    print("확장 토큰:", output["expanded_tokens"])
    print("감지 의도:", output["query_intents"])
    print("-" * 80)
    for index, row in enumerate(output["results"], start=1):
        print(f"{index}. {row['메뉴명']} ({row['카테고리']}) - {row['최종점수'] * 100:.1f}점")
        print(f"   TF-IDF: {row['TFIDF점수'] * 100:.1f}점 / 의도: {row['의도점수'] * 100:.1f}점 / 매칭: {row['매칭의도']}")
        print(f"   설명: {row['설명']}")
        print()

print_recommendations("해장 잘되는 음식")

In [ ]:
print_recommendations("칼칼한 음식")

In [ ]:
print_recommendations("다이어트에 좋은 음식")

In [ ]:
print_recommendations("고소하고 부드러운 음식")

## 10. 실습 해석

이번 실습의 핵심은 다음과 같습니다.

- CSV의 `설명` 컬럼을 하나의 문서로 보고 TF-IDF를 계산할 수 있다.
- 사용자 입력도 하나의 짧은 문서로 보고 메뉴 설명과 유사도를 비교할 수 있다.
- 한국어에서는 `다이어트에`, `칼칼한`처럼 조사와 어미가 붙기 때문에 간단한 정규화만 해도 결과가 개선된다.
- `해장`처럼 메뉴 설명에 직접 등장하지 않는 의도는 도메인 키워드 확장을 통해 추천 품질을 보정할 수 있다.

실제 서비스에서는 식당 메뉴판 추천, 배달앱 메뉴 검색, 리뷰 문장 기반 메뉴 태그 자동 생성으로 확장할 수 있습니다.

## 11. Next.js 확장 버전

이 노트북의 로직은 아래 Next.js 앱으로도 구현되어 있습니다.

```bash
cd ../nextjs-menu-recommender
npm install
npm run dev -- --port 3012
```

로컬 URL:

```text
http://localhost:3012
```

수업용 Python 노트북은 개념 이해와 제출용으로 사용하고, Next.js 앱은 포트폴리오용 서비스 화면으로 발전시키면 좋습니다.